# FineWeb-Edu GPT — Analysis

Training now happens in `train.py`:

    uv run python projects/fineweb-edu/gpt/train.py

A decoder-only Transformer (grouped-query attention + RoPE + QK-norm) pretrained on
[FineWeb-Edu](https://huggingface.co/datasets/HuggingFaceFW/fineweb-edu) `sample-10BT`,
tokenized with the pretrained [LiquidAI/LFM2.5-230M](https://huggingface.co/LiquidAI/LFM2.5-230M)
subword tokenizer, optimized with Muon + AdamW and Cut Cross Entropy.

This notebook pulls model config and training curves for any wandb run (set
`WANDB_RUN` below) and loads the local checkpoint for text generation.

In [ ]:
%matplotlib inline
%config InlineBackend.figure_formats = ['retina', 'svg']
%load_ext autoreload
%autoreload 2

import os
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd
import torch
import wandb

from chimera.data import FineWebEduDataModule
from chimera.models import GPT

# datasets + tokenizer caches live on the big volume
os.environ.setdefault("HF_HOME", "/mnt/ai/data/hf")

RUN_DIR = "/mnt/ai/runs/fineweb-edu/gpt"
DATA_DIR = "/mnt/ai/data"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

WANDB_ENTITY = "karanravindra"
WANDB_PROJECT = "fineweb-edu-gpt"
# Set to a run id (or full "entity/project/run_id" path) to analyze a specific
# run; leave None to fall back to the most recently created run in the project.
WANDB_RUN = None

## Pick a wandb run

`train.py` logs its full argparse config (`n_embd`, `seq_len`, muP knobs, ...) to
wandb, so the model shape and metrics below are read straight from the selected
run instead of hardcoded constants -- point `WANDB_RUN` at any run (e.g. an
`--arch` sweep member) and this notebook adapts to it.

In [ ]:
api = wandb.Api()
latest_run = api.runs(f"{WANDB_ENTITY}/{WANDB_PROJECT}", order="-created_at")[0]
run = (
    api.run(
        WANDB_RUN if "/" in WANDB_RUN else f"{WANDB_ENTITY}/{WANDB_PROJECT}/{WANDB_RUN}"
    )
    if WANDB_RUN
    else latest_run
)
cfg = run.config
print(f"run: {run.name} ({run.id})  state={run.state}  url={run.url}")

# Falls back to train.py's own defaults for older runs predating hyperparam logging.
SEQ_LEN = cfg.get("seq_len", 2048)
N_EMBD = cfg.get("n_embd", 384)
N_HEAD = cfg.get("n_head", 12)
N_KV_HEAD = cfg.get("n_kv_head", 3)
N_LAYER = cfg.get("n_layer", 6)
TIE_EMBEDDING = cfg.get("tie_embedding", True)
MAX_TRAIN_TOKENS = cfg.get("max_train_tokens", 1_000_000_000)
print(
    f"n_embd={N_EMBD} n_head={N_HEAD} n_kv_head={N_KV_HEAD} n_layer={N_LAYER} "
    f"seq_len={SEQ_LEN} tie_embedding={TIE_EMBEDDING}"
)

if run.id != latest_run.id:
    print(
        f"\nNOTE: train.py's ModelCheckpoint always overwrites "
        f"{RUN_DIR}/checkpoints/gpt.ckpt, so the 'Load checkpoint' section below "
        f"loads weights from the most recent LOCAL run ({latest_run.name}, "
        f"{latest_run.id}) regardless of which run is selected here -- they only "
        f"match if WANDB_RUN is None or already points at the latest run. Config "
        f"and training curves elsewhere in this notebook ARE specific to "
        f"{run.name} ({run.id})."
    )

## Data

Load FineWeb-Edu `sample-10BT` with the fixed **LiquidAI/LFM2.5-230M** tokenizer.
Documents are concatenated with `<|endoftext|>` appended after each, so the model
learns document boundaries instead of attending across unrelated pages.

In [ ]:
dm = FineWebEduDataModule(
    data_dir=DATA_DIR,
    name="sample-10BT",
    batch_size=8,
    seq_len=SEQ_LEN,
    tokenizer_backend="pretrained",  # LiquidAI/LFM2.5-230M
    add_eos=True,
    max_train_tokens=MAX_TRAIN_TOKENS,
    num_workers=4,
)
dm.prepare_data()
dm.setup("fit")

print(f"tokenizer={dm.pretrained_id}  vocab_size={dm.vocab_size}")
print(f"eos_token={dm.eos_token!r}  eos_id={dm.eos_id}")
print(f"train chunks={len(dm.train_dataset):,}  val chunks={len(dm.val_dataset):,}")

x, y = next(iter(dm.train_dataloader()))
print("sample:", repr(dm.decode(x[0][:64])))
# confirm the document separator is actually present in the stream
n_eos = int((x == dm.eos_id).sum())
print(f"<|endoftext|> tokens in one batch of {x.numel():,}: {n_eos}")

# most frequent tokens in one batch, shown as their decoded text
counts = Counter(x.flatten().tolist())
top = counts.most_common(20)
labels = [repr(dm.decode([tid])) for tid, _ in top]
values = [c for _, c in top]

plt.figure(figsize=(12, 4))
plt.bar(range(len(top)), values)
plt.title("Top-20 Tokens (one training batch)")
plt.xlabel("Token")
plt.ylabel("Count")
plt.xticks(range(len(top)), labels, rotation=60, ha="right")
plt.tight_layout()
plt.show()

## Load checkpoint (most recent local run)

`train.py`'s `ModelCheckpoint` always writes to the same path, so only the
weights from the most recently *trained-on-this-machine* run survive on disk --
unlike config/metrics, they can't be pulled for an arbitrary wandb run. See the
NOTE printed above if the selected run differs from the latest local one.

In [ ]:
ckpt = torch.load(
    f"{RUN_DIR}/checkpoints/gpt.ckpt", map_location="cpu", weights_only=False
)
# train.py torch.compile()s the model, so state_dict keys are prefixed
# "model._orig_mod." (compiled) or "model." (eager) -- strip both.
model_state = {
    k.removeprefix("model.").removeprefix("_orig_mod."): v
    for k, v in ckpt["state_dict"].items()
    if k.startswith("model.")
}

model = GPT(
    vocab_size=dm.vocab_size,
    block_size=SEQ_LEN,
    n_embd=N_EMBD,
    n_head=N_HEAD,
    n_kv_head=N_KV_HEAD,
    n_layer=N_LAYER,
    tie_embedding=TIE_EMBEDDING,
)
model.load_state_dict(model_state)
model.to(DEVICE).eval()
print(
    f"loaded GPT ({sum(p.numel() for p in model.parameters()) / 1e6:.2f}M params, "
    f"epoch {ckpt['epoch']}, step {ckpt['global_step']})"
)

## Training curves

Pulled straight from the selected wandb run's history (`run.scan_history()`),
so it reflects whichever run was picked above -- not just whatever happened to
train most recently on this machine. Validation is logged under both `val/*`
(during `fit`) and `test/*` (the final `trainer.test`).

In [ ]:
metrics = pd.DataFrame(run.scan_history())

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle(f"Training Metrics — {run.name} ({run.id})")
for ax, key, title in zip(axes, ["loss", "bpt"], ["Loss (nats)", "Bits per Token"]):
    for stage, style in [
        ("train", {}),
        ("val", {"marker": "o", "ls": ""}),
        ("test", {"marker": "s", "ls": ""}),
    ]:
        col = f"{stage}/{key}"
        if col not in metrics.columns:
            continue
        d = metrics.dropna(subset=[col])
        if not len(d):
            continue
        ax.plot(d["_step"], d[col], label=stage, **style)
    ax.set_title(title)
    ax.set_xlabel("Step")
    ax.set_ylabel(title)
    ax.legend()
    ax.grid(alpha=0.3)
plt.show()

## Text generation

In [ ]:
prompt = (
    "Question: How is bipolar disorder different from unipolar depression or 'regular' depression?\n"
    + "Answer: Bipolar disorder, also known as manic-depressive illness, is a mental health condition characterized by extreme mood swings that include emotional highs (mania or hypomania) and lows (depression). In contrast, unipolar depression, often referred to as major depressive disorder, involves persistent feelings of sadness or a lack of interest in external stimuli, without the manic episodes seen in bipolar disorder.\n\n"
    + "Question: What are the common symptoms of bipolar disorder?\n"
)
idx = torch.tensor([dm.tokenizer.encode(prompt)], device=DEVICE)

# Eager by default (no compile warmup). Pass compile=True for ~2x faster decode
# when generating repeatedly -- it costs a one-time ~10s compile on the first call.
generated = model.generate(idx, max_new_tokens=200, temperature=0.8)
print(dm.decode(generated[0].cpu()))

In [ ]:
prompt = """It was purchased for $4,345.00 in 1890, a lot of money back then, and used in the 1906 earthquake."""

idx = torch.tensor([dm.tokenizer.encode(prompt)], device=DEVICE)
generated = model.generate(idx, max_new_tokens=200, temperature=1.2)
print(dm.decode(generated[0].cpu()))